# **BUILDING CLAUDE MANAGED AGENTS**

- Tutorial to learn how to build claude managed agents with Anthropic's Python SDK, cloud sandbox, tools, skills, file uploads, session managment, code execution, and automated Excel report generation.
- Managed clients are built around four main parts:
    1. **Agent:** the reusable configuration that contains the model, system prompt, tools, MCP servers, and skills.
    2. **Environment:** the location where the agent runs, either in Anthropic's cloud sandbox or in self-hosted sandbox.
    3. **Session:** a running instance of the agent that performs a specific task.
    4. **Events:** the messages, tool calls, results, and status updates exchanged whle the session runs. 


### **2. Setting up Anthropic SDK**
- Install Anthropic Python SDK from which we create agents, environments, files, and sessions. 

In [3]:
%pip install -q --upgrade anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
## Loading API Key from .env file into my code
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

In [5]:
## Laoding the anthropic API key from the environment variable
import os
from anthropic import Anthropic

api_key = os.environ.get("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in your environment or in a local .env file."

In [6]:
## setting up the BETA version of the API
BETA_FLAG = "managed-agents-2026-04-01"

client = Anthropic(api_key=api_key)

In [7]:
## creating a dictionary to store IDs of every resource created in this notebook, so that we can clean up after ourselves
## Resources include the agent, environment, uploaded files, conversations, and the session. 

created = {"agent": None, "environment": None, "file": None, "session": None}
print("✓ Anthropic client initialized.")

✓ Anthropic client initialized.


#### **3. Creating the Managed Agent**
- A managed agent is the reusable configuration for my workflow.
- When creating a managed agent, choose Claude Model, write a system prompt defining roles and instructions, while attaching the tools and skills it can use. 
- Same agent can be reused for multiple sessions instead of recreating the configuration each time.
- I have renamed my agent to **Sonnet 5 Data Analyst** and will use the claude-sonnet-5 model.
- The system prompt tells the agent to behave like a careful data analyst: inspecting files mounted in the folder /workspace, use code execution for analysis, keep results concise, and save final files to /mnt/session/outputs.

In [8]:
agent = client.beta.agents.create(
    name="Sonnet 5 Data Analyst",
    model="claude-sonnet-5",
    system=(
        "You are a meticulous data analyst. When asked about data, always read the "
        "file mounted at /workspace, analyse it with the code execution tool, and "
        "report concise, numeric results. Use the XLSX skill for spreadsheet work. "
        "Save final artifacts to /mnt/session/outputs."
    ),
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
    skills=[{"type": "anthropic", "skill_id": "xlsx"}],
)

- The *agent_toolset_20260401* gives agent access to Anthropic's built-in tools, and the xlsx skill provides expert guidance for creating and analyzing Excel workbooks. 
- Next, we save agent ID to be used later as below. 

In [9]:
created["agent"] = agent.id
print(f"✓ Created agent: {agent.id}")

✓ Created agent: agent_01YLJEEKWdETX7L7ML53YfPb


#### **4. Configure the Anthropic Cloud Sandbox**
- We create an environment within which the Managed Agents runs during a session/task.
- An environment acts as safe and secure sandbox which gives the agent a sepparate workplace to read mounted files, write code, and run commands. 
- Our environment will be created using Anthropic's cloud environment. 

In [10]:
environment = client.beta.environments.create(
    name="code-exec-sandbox",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

created["environment"] = environment.id
print(f"✓ Created environment: {environment.id}")

✓ Created environment: env_01BSQG4JJXsY1DrMDE6Q4DnQ


#### **5. Upload Data with the Anthropic Files API**
- Next step, upload the dataset for the agent to analyze. Our **Sonnet 5 Data Analyst** uses Anthropic FIles API to be mounted inside session environment.
- After uploading the dataset, we check if file exists and confirm that it contains the expected number of data rows. 

In [14]:
from pathlib import Path

csv_path = Path("parental_leave.csv")
assert csv_path.exists(), f"Missing input file: {csv_path.resolve()}"

row_count = sum(1 for _ in csv_path.open(encoding="latin-1")) - 1
assert row_count == 1601, f"Expected 1601 data rows, found {row_count}"

- Next upload the file and store its ID for later cleanup.

In [15]:
uploaded = client.beta.files.upload(file=csv_path)

created["file"] = uploaded.id
print(f"✓ Uploaded {csv_path}: {uploaded.id} ({row_count} rows)")

✓ Uploaded parental_leave.csv: file_01UWgX8YH82z488c9g8vfhsD (1601 rows)


#### **6. Initialize an Agent Execution Session**
- Next, we create a session. A session connects the agent, the environment, and the resources it needs for a specific task.
- We then will mount the uploaded CSV file at /workspace/parental_leave.csv, so that the agent accesses it from inside the sandbox. 

In [16]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    resources=[
        {
            "type": "file",
            "file_id": uploaded.id,
            "mount_path": "/workspace/parental_leave.csv",
        },
    ],
)

created["session"] = session.id
print(f"✓ Created session: {session.id}")

✓ Created session: sesn_01WuLVwd3yHvk5qs2XQCYxpj


#### **7. Stream the Agent’s Response**
- Next step, we send task to the session and stream agent's activity as it works.
- Remember, creating a session only prepares agent and sandbox. The agent only starts working after it receives a user.message event. 
- The event stream lets us see the agent's messages, tool calls, and final session status in real time. 
- We add a prompt that tells the agent to analyze mounted CSV file, write, and run a Python Script, create a JSON summary, and building an Excel report. 
- Lastly, we create three variables to collect the agent's task, record the tools it uses, and confirm whether the session finishes successfully. 

In [17]:
agent_text_parts = []
tools_used = []
final_status = None

with client.beta.sessions.events.stream(session.id) as stream:
    # Send the user message once the stream is open.
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Use the XLSX skill and analyze /workspace/parental_leave.csv. "
                            "Write /mnt/session/outputs/analyze_data.py, run that Python "
                            "script, and have it create /mnt/session/outputs/summary.json "
                            "containing: total companies analyzed, average paid/unpaid "
                            "maternity and paternity leave, industry-level aggregations "
                            "(mean, min, max for each leave type), and the top 5 companies "
                            "by total paid leave. "
                            "Also create /mnt/session/outputs/parental_leave_report.xlsx with: "
                            "(1) a 'Source Data' sheet with the raw dataset, "
                            "(2) an 'Industry Summary' sheet with average leave by industry, "
                            "(3) a 'Leave Gap Analysis' sheet comparing paid maternity vs "
                            "paid paternity leave by company, and "
                            "(4) a 'Summary' sheet with overall dataset statistics."
                        ),
                    }
                ],
            }
        ],
    )

    for event in stream:
        etype = getattr(event, "type", None)

        if etype == "agent.message":
            for block in event.content:
                txt = getattr(block, "text", None)
                if txt:
                    print(txt, end="")
                    agent_text_parts.append(txt)

        elif etype == "agent.tool_use":
            name = getattr(event, "name", "<tool>")
            print(f"\n[tool_use] {name}")
            tools_used.append(name)

        elif etype == "session.status_idle":
            final_status = "idle"
            print("\n\n✓ Agent finished; session is idle.")
            break

        elif etype == "session.status_error":
            final_status = "error"
            print("\n✗ Session reported an error.")
            break

print("Tools used:", tools_used)


[tool_use] read

[tool_use] bash

[tool_use] bash

[tool_use] bash
Good, found the file. Let's check for trailing empty columns, data types, missing values, and get a full picture before writing the script.
[tool_use] bash

[tool_use] bash

[tool_use] bash

[tool_use] bash
Now I have a clear picture of the data. Let me write the analysis script.
[tool_use] bash

[tool_use] bash

[tool_use] write

[tool_use] bash

[tool_use] bash

[tool_use] bash

[tool_use] bash

[tool_use] edit
Also update the copy in outputs since the script uses the local copy path, and re-copy the source file already there. Now re-run:
[tool_use] bash
Now let's recalc the workbook with LibreOffice as required by the skill, then verify.
[tool_use] bash
No formula errors. Now let's verify the actual values by loading with data_only=True and spot-checking against pandas-computed values.
[tool_use] bash
Values match pandas computations exactly. Let's cross-check against summary.json and validate the other sheets too.


- During the run, we see tool events such as **read**,**bash**, **write**, and **edit**.
- The run shows that the agent is inspecting available instructions and files, writing the analysis script, running activities inside the sandbox, and correcting the issues it finds. 
- Once agent finishes the work, it says **agent finished: session is idle**. 
- Anthropic manages execution for its built-in tools inside the sandbox; you only need to handle tool results yourself when using custom tools. 

#### **8. Retrieve Generated Files and Event History**
- After the session is complete, we then inspect its saved event history and download the files created by the agent. 
- The event history provides a full record of the session, including model requests, tool calls, tool results, and status changes. 

In [18]:
history = client.beta.sessions.events.list(session.id, order="asc")

print("--- Session event history ---")
for event in history.data:
    print(event.type)

print(f"({len(history.data)} events total)")

--- Session event history ---
session.status_running
session.thread_status_running
user.message
span.model_request_start
agent.thinking
agent.tool_use
agent.tool_use
span.model_request_end
agent.tool_result
agent.tool_result
span.model_request_start
agent.thinking
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.thinking
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.message
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.thinking
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.message
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.tool_use
span.model_request_end
agent.tool_result
span.model_request_start
agent.thinking
agent.tool_use

- Next step, we list the files attached to the session and download any files marked as downloadable. The files are saved locally in an outputs folder. 

In [19]:
import os

os.makedirs("outputs", exist_ok=True)

files = client.beta.files.list(scope_id=session.id, betas=[BETA_FLAG])
downloadable = [f for f in files.data if f.downloadable]

print(f"Found {len(downloadable)} downloadable file(s) for this session.")

downloaded_paths = []

for f in downloadable:
    try:
        content = client.beta.files.download(f.id, betas=[BETA_FLAG])
        local_path = os.path.join("outputs", f.filename)

        content.write_to_file(local_path)
        downloaded_paths.append(local_path)

        print(f"  downloaded {f.id} -> {local_path}")
    except Exception as exc:
        print(f"  skip {f.id}: {exc}")

Found 4 downloadable file(s) for this session.
  downloaded file_011CeS8iptXYwJXdptZ6t85g -> outputs\parental_leave_report.xlsx
  downloaded file_011CeS8i717NfAN7JR5Mucbh -> outputs\summary.json
  downloaded file_011CeS8i4dUzR4syLMtLiVoX -> outputs\analyze_data.py
  downloaded file_011CeS8XUFEx593669cHmrdH -> outputs\parental_leave.csv


- Checking that all expected outputs were downloaded successfully. 

In [21]:
expected_outputs = {"analyze_data.py", "summary.json", "parental_leave_report.xlsx"}
downloaded_names = {os.path.basename(path) for path in downloaded_paths}

assert expected_outputs <= downloaded_names, (
    f"Missing expected outputs: {sorted(expected_outputs - downloaded_names)}"
)

#### **9. Analyze the Python, JSON, and Excel Outputs**
- After the agent completed the task, the generated files are downloaded into the local outputs/ directory.
- These files show that the agent did more than return a text response: it wrote and executed code, created structured data, and produced a spreadsheet report that can be reviewed independently.

#### **10. Clean Up Managed Agent API Resources**
- Managed Agent resources remain available until you remove them, so it is important to clean them up once the task is complete. 
- Active resources, especially running sessions and environments, can continue to incur costs if they are left running.
- A session must be idle before it can be deleted. 
- In this final step, we delete the session, uploaded file, and environment, then archive the agent. 
- Each cleanup action is wrapped in a helper function so that one failed deletion does not stop the remaining resources from being removed.

In [22]:
def safe(label, fn):
    try:
        fn()
        print(f"✓ deleted {label}")
    except Exception as exc:
        print(f"· could not delete {label}: {exc}")

if created["session"]:
    safe("session", lambda: client.beta.sessions.delete(created["session"]))

if created["file"]:
    safe("file", lambda: client.beta.files.delete(created["file"]))

if created["environment"]:
    safe("environment", lambda: client.beta.environments.delete(created["environment"]))

if created["agent"]:
    safe("agent (archived)", lambda: client.beta.agents.archive(created["agent"]))

print("\n🎉 Cleanup complete. The agent is archived; other resources were deleted.")

✓ deleted session
✓ deleted file
✓ deleted environment
✓ deleted agent (archived)

🎉 Cleanup complete. The agent is archived; other resources were deleted.
